# TP05 — Projet final Deep Learning

## Contexte
Ce notebook correspond au **projet final de synthèse** du module *Machine Learning - Deep Learning*.

Pendant la semaine, vous avez travaillé sur :
- **Jour 1** : MLP sur données tabulaires ;
- **Jour 2** : diagnostic d'apprentissage, underfitting, overfitting, régularisation ;
- **Jour 3** : CNN pour classification d'images ;
- **Jour 4** : RNN / GRU / LSTM pour séries temporelles.

Aujourd'hui(Imaginons que vous lisiez ceci vendredi, et non jeudi après-midi :P), vous devez construire un **mini-projet de groupe** de bout en bout :
- chargement des données ;
- preprocessing ;
- choix du modèle ;
- entraînement ;
- validation ;
- évaluation ;
- analyse d'erreurs ;
- amélioration ou comparaison ;
- sauvegarde ;
- rechargement ;
- inférence ;
- mini-démo dans le notebook.

## Consignes importantes
- Chaque groupe choisit **un seul parcours** :
  - **A** : classification d'images avec CNN ;
  - **B** : séries temporelles avec RNN / GRU / LSTM ;
  - **C** : données tabulaires avec MLP.
- Une fois le parcours choisi, vous pouvez **supprimer** ou **ignorer** les blocs des autres parcours.
- La **WebApp n'est pas obligatoire**.
- La partie "mise en production" est **simplifiée** :
  - sauvegarder le modèle ;
  - recharger le modèle ;
  - faire une prédiction sur une nouvelle donnée ;
  - préparer une mini-démo d'inférence dans le notebook.
- **Gradio** est uniquement un **bonus facultatif**.
- Pas de backend, pas de frontend, pas de Flask, pas de FastAPI, pas de Docker.
- Le notebook doit rester **simple, propre, exécutable dans Google Colab**.

## Règle importante sur le choix du dataset
Les étudiants peuvent choisir eux-mêmes leur dataset, à condition qu'il respecte les contraintes du TP :
- dataset léger ;
- accessible facilement dans Google Colab ;
- pas d'API payante ;
- pas de compte externe obligatoire ;
- temps de chargement et d'entraînement raisonnable ;
- problème compatible avec le parcours choisi ;
- données suffisamment simples pour permettre une analyse en une journée.

**Les datasets proposés sont des suggestions. Vous pouvez choisir un autre dataset, à condition de justifier votre choix et de respecter les contraintes de temps, de simplicité et de faisabilité du projet. L’objectif du Jour 5 est d’adapter les méthodes vues pendant la semaine à un nouveau problème, pas de recopier un ancien TP.**

## TODO groupe
- Choisir un parcours réaliste.
- Choisir un dataset faisable en une journée.
- Répartir le travail.
- Documenter les choix dans les cellules Markdown prévues.


## 2. Informations groupe

**Nom du groupe :** ...

**Étudiant 1 :** JARI Salah Eddine

**Étudiant 2 :** Elyas Abdenbi

**Étudiant 3 (si applicable) :** ...

**Parcours choisi :** C

**Parcours choisi :** C

- Dataset choisi : Wine dataset (sklearn.datasets.load_wine)
- Problème traité : Classification de vins en 3 catégories à partir de mesures chimiques
- Variable cible : classe du vin (0, 1 ou 2)
- Type de tâche : classification multi-classe
- Métrique principale : accuracy
- Baseline ou référence simple : classificateur majoritaire (~40 %) ou logistic regression sklearn
- Critère de réussite minimal : accuracy test > 90 %
- Limite principale attendue : dataset très petit (178 échantillons), risque de sur-apprentissage rapide


## 3. Choix du parcours

Choisissez **un seul** parcours :
- `TRACK = "A"` pour la classification d'images avec CNN ;
- `TRACK = "B"` pour les séries temporelles ;
- `TRACK = "C"` pour les données tabulaires avec MLP.

> **TODO étudiant** : une fois le choix stabilisé, vous pouvez supprimer ou ignorer les autres blocs.


In [ ]:
TRACK = "C"   # Remplacer par "A", "B" ou "C"
print("Parcours choisi :", TRACK)
assert TRACK in ["A", "B", "C"], "TRACK doit être 'A', 'B' ou 'C'"


: 

## 4. Imports et configuration

Le notebook utilise uniquement des dépendances compatibles avec **Google Colab** :
- `tensorflow / keras`
- `numpy`
- `pandas`
- `matplotlib`
- `sklearn`

Aucune dépendance exotique. Aucun compte externe.


In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    ConfusionMatrixDisplay,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.rcParams["figure.figsize"] = (12, 4)

print("TensorFlow :", tf.__version__)
print("Keras backend :", keras.backend.backend())
print("Setup OK")


: 

## 5. Chargement des données selon le parcours

### Règle générale
Les datasets mentionnés ci-dessous ne sont **pas obligatoires**. Ce sont des **options recommandées** ou des **datasets de secours**.

### Parcours A — Image classification
- **Option recommandée** : **CIFAR-10** ;

### Parcours B — Time series
- **Option recommandée** : série temporelle synthétique simple avec **contexte métier clair**, par exemple :
  - demande énergétique,
  - trafic web,
  - fréquentation magasin.

### Parcours C — Données tabulaires
- **Option recommandée** : **Wine dataset** de `sklearn` ;
- autre dataset `sklearn` possible si le groupe le justifie.

### Si vous choisissez votre propre dataset
Vous devez expliquer dans le **README** :
- pourquoi ce dataset a été choisi ;
- quel est le problème traité ;
- quelle est la variable cible ;
- quelle métrique principale est utilisée ;
- pourquoi le modèle choisi est adapté aux données.

> **TODO étudiant** : vérifier que le dataset choisi reste raisonnable à charger et à entraîner en CPU dans Google Colab.


In [ ]:
if TRACK == "A":
    # Option recommandée par défaut : CIFAR-10
    # Fallback possible si nécessaire : Fashion-MNIST
    from tensorflow.keras.datasets import cifar10

    class_names = [
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck"
    ]

    (x_train_full, y_train_full), (x_test_raw, y_test) = cifar10.load_data()
    y_train_full = y_train_full.ravel()
    y_test = y_test.ravel()

    # Sous-ensemble raisonnable pour Colab CPU
    x_train_raw = x_train_full[:15000]
    y_train = y_train_full[:15000]
    x_test_raw = x_test_raw[:3000]
    y_test = y_test[:3000]

    print("Train images :", x_train_raw.shape)
    print("Test images  :", x_test_raw.shape)

elif TRACK == "B":
    # Exemple métier simplifié : demande énergétique horaire simulée
    BUSINESS_CONTEXT = "Demande énergétique"

    def generate_time_series(n_steps=2400, seed=42):
        rng = np.random.default_rng(seed)
        t = np.arange(n_steps)
        trend = 0.015 * t
        seasonal_short = 3.0 * np.sin(2 * np.pi * t / 24)
        seasonal_long = 1.8 * np.sin(2 * np.pi * t / 168)
        noise = rng.normal(0.0, 0.8, n_steps)
        base_signal = 10 + trend + seasonal_short + seasonal_long + noise

        series = np.zeros(n_steps, dtype=np.float32)
        series[:24] = base_signal[:24]

        for i in range(24, n_steps):
            nonlinear_term = 0.15 * np.tanh(series[i - 1] - series[i - 24])
            series[i] = (
                0.45 * series[i - 1]
                + 0.15 * series[i - 24]
                + 0.35 * base_signal[i]
                + nonlinear_term
            )
        return series

    series = generate_time_series(n_steps=2400, seed=SEED)
    print("Contexte métier :", BUSINESS_CONTEXT)
    print("Longueur série :", len(series))

elif TRACK == "C":
    # Option recommandée par défaut : Wine dataset
    from sklearn.datasets import load_wine

    data = load_wine()
    X_df = pd.DataFrame(data.data, columns=data.feature_names)
    y = pd.Series(data.target, name="target")
    target_names = list(data.target_names)

    print("Shape X :", X_df.shape)
    print("Distribution y :")
    print(y.value_counts().sort_index())


## 6. Visualisation rapide

Objectif : regarder rapidement les données avant toute modélisation.

> **TODO étudiant** : commenter en 3 lignes ce que vous observez sur vos données.
> Si vous avez remplacé le dataset recommandé par un autre, expliquez ici pourquoi ce choix est pertinent.


In [ ]:
if TRACK == "A":
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    axes = axes.ravel()
    for i, ax in enumerate(axes):
        ax.imshow(x_train_raw[i])
        ax.set_title(class_names[y_train[i]])
        ax.axis("off")
    plt.tight_layout()
    plt.show()

elif TRACK == "B":
    plt.figure(figsize=(14, 4))
    plt.plot(series)
    plt.title("Série temporelle synthétique")
    plt.xlabel("Temps")
    plt.ylabel("Valeur")
    plt.show()

elif TRACK == "C":
    display(X_df.head())
    print("Noms des classes :", target_names)
    print("Résumé statistique rapide :")
    display(X_df.describe().T.head())


**Commentaire groupe :**

- **Ce qu'on observe :** 13 features chimiques continues, sans valeurs manquantes. Les distributions varient fortement selon les classes — par exemple `proline` est ~1000 pour la classe 0 et ~520 pour la classe 2, et `flavanoids` est très discriminant. Les classes sont relativement équilibrées (59 / 71 / 48 échantillons).
- **Difficulté anticipée :** Le dataset est très petit (178 échantillons). Le risque principal est le sur-apprentissage rapide, surtout avec un MLP dont la capacité dépasse ce que les données peuvent soutenir. Certaines features sont également corrélées entre elles.
- **Cohérence du modèle :** Un MLP est naturellement adapté aux données tabulaires numériques continues. La standardisation des features assure une convergence stable et équilibrée des gradients.


## 7. Preprocessing selon le parcours

Rappel :
- pas de fuite d'information ;
- split cohérent ;
- normalisation / standardisation propre ;
- pour les séries temporelles : **pas de shuffle**, scaler fit uniquement sur train.


In [ ]:
if TRACK == "A":
    x_train_raw = x_train_raw.astype("float32") / 255.0
    x_test = x_test_raw.astype("float32") / 255.0

    x_train = x_train_raw[:12000]
    y_train_final = y_train[:12000]
    x_val = x_train_raw[12000:15000]
    y_val = y_train[12000:15000]

    print("x_train :", x_train.shape)
    print("x_val   :", x_val.shape)
    print("x_test  :", x_test.shape)

elif TRACK == "B":
    train_end = int(len(series) * 0.70)
    val_end = int(len(series) * 0.85)

    scaler = StandardScaler()
    scaler.fit(series[:train_end].reshape(-1, 1))
    series_scaled = scaler.transform(series.reshape(-1, 1)).ravel()

    LOOKBACK = 48
    HORIZON = 1

    def make_windows(values_scaled, start_idx, end_idx, lookback=48, horizon=1):
        Xw, yw = [], []
        for target_start in range(start_idx, end_idx - horizon + 1):
            Xw.append(values_scaled[target_start - lookback:target_start])
            yw.append(values_scaled[target_start:target_start + horizon])
        return np.array(Xw, dtype=np.float32), np.array(yw, dtype=np.float32)

    X_train, y_train_final = make_windows(series_scaled, LOOKBACK, train_end, LOOKBACK, HORIZON)
    X_val, y_val = make_windows(series_scaled, train_end, val_end, LOOKBACK, HORIZON)
    X_test, y_test = make_windows(series_scaled, val_end, len(series), LOOKBACK, HORIZON)

    X_train = np.expand_dims(X_train, axis=-1)
    X_val = np.expand_dims(X_val, axis=-1)
    X_test = np.expand_dims(X_test, axis=-1)

    print("X_train :", X_train.shape)
    print("X_val   :", X_val.shape)
    print("X_test  :", X_test.shape)

elif TRACK == "C":
    X_train_df, X_test_df, y_train_s, y_test = train_test_split(
        X_df, y, test_size=0.2, random_state=SEED, stratify=y
    )
    X_train_df, X_val_df, y_train_s, y_val = train_test_split(
        X_train_df, y_train_s, test_size=0.2, random_state=SEED, stratify=y_train_s
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_df)
    X_val = scaler.transform(X_val_df)
    X_test = scaler.transform(X_test_df)
    y_train_final = y_train_s.values
    y_val = y_val.values
    y_test = y_test.values

    print("X_train :", X_train.shape)
    print("X_val   :", X_val.shape)
    print("X_test  :", X_test.shape)


**Validation groupe :**

- **Transformation appliquée :** `StandardScaler` (centrage-réduction) : on soustrait la moyenne et on divise par l'écart-type, feature par feature. Le scaler est fitté **uniquement sur le train set** (`.fit_transform()`), puis appliqué (`.transform()`) sur val et test.
- **Pourquoi est-elle nécessaire ?** Les 13 features ont des échelles très différentes (ex. `proline` ~700 vs `nonflavanoid_phenols` ~0.3). Sans normalisation, les features à grande amplitude dominent les gradients, ce qui ralentit ou déstabilise la convergence du MLP.
- **Risque de data leakage :** Si le `StandardScaler` était fitté sur l'ensemble du dataset avant le split (train + val + test), les statistiques des données de test contamineraient l'entraînement. L'estimation des performances serait alors trop optimiste et non représentative du déploiement réel.


## 8. Construction du modèle selon le parcours

Le modèle ci-dessous est une **baseline raisonnable**. Il peut ensuite être amélioré dans la section 13.

> **TODO étudiant** : si vous utilisez un dataset différent de celui recommandé, adaptez les shapes d'entrée, la couche de sortie et la métrique principale.


In [ ]:
if TRACK == "A":
    model = keras.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ])
    loss = "sparse_categorical_crossentropy"
    metrics = ["accuracy"]

elif TRACK == "B":
    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1], 1)),
        layers.GRU(64),
        layers.Dense(1)
    ])
    loss = "mse"
    metrics = [keras.metrics.MeanAbsoluteError(name="mae")]

elif TRACK == "C":
    model = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(32, activation="relu"),
        layers.Dense(len(target_names), activation="softmax")
    ])
    loss = "sparse_categorical_crossentropy"
    metrics = ["accuracy"]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss,
    metrics=metrics
)

model.summary()


## 9. Entraînement

On entraîne une première version du modèle avec **EarlyStopping** pour garder un temps raisonnable.

> **TODO étudiant** : noter si vous observez un apprentissage stable, trop lent ou instable.


In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

start_time = time.perf_counter()

history = model.fit(
    X_train, y_train_final,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop]
)

train_time = time.perf_counter() - start_time
print(f"Temps d'entraînement : {train_time:.2f} s")


## 10. Courbes d'apprentissage

Les courbes doivent être commentées :
- apprentissage correct ?
- overfitting ?
- underfitting ?


In [ ]:
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(history_df["loss"], label="train")
axes[0].plot(history_df["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

metric_columns = [c for c in history_df.columns if c not in ["loss", "val_loss"]]
if len(metric_columns) > 0:
    metric_name = metric_columns[0]
    val_metric_name = "val_" + metric_name
    axes[1].plot(history_df[metric_name], label="train")
    axes[1].plot(history_df[val_metric_name], label="val")
    axes[1].set_title(metric_name)
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
else:
    axes[1].axis("off")

plt.tight_layout()
plt.show()


**Commentaire groupe :**

- **Description des courbes :** La train loss part de 1.17 et descend régulièrement jusqu'à 0.17 en 15 epochs. La val loss part de 0.98 et atteint 0.19 — les deux courbes convergent sans divergence. L'EarlyStopping n'a pas été déclenché : le modèle a bénéficié des 15 epochs complètes.
- **Validation accuracy > train accuracy ?** Oui, pendant la majorité de l'entraînement. C'est un effet normal du **Dropout** : il est actif pendant le training (désactive aléatoirement des neurones) mais inactif pendant la validation. Ce n'est **pas** du sur-apprentissage.
- **Pas d'overfitting observé :** Les deux courbes de loss décroissent de façon monotone et convergent. Accuracy finale : train = 99.1 %, val = 96.6 %.
- **Piste d'amélioration :** Tester un dropout plus élevé ou une architecture plus profonde (exploré en §13).


## 11. Évaluation

On calcule maintenant les métriques principales sur le test set.

> **TODO étudiant** : si vous avez changé de dataset, vérifiez que la métrique affichée reste cohérente avec votre problème et votre variable cible.


In [ ]:
if TRACK in ["A", "C"]:
    y_pred_prob = model.predict(X_test, verbose=0)

    if TRACK == "A":
        y_pred = np.argmax(y_pred_prob, axis=1)
        score_main = accuracy_score(y_test, y_pred)
        print("Accuracy test :", round(score_main, 4))
        print(classification_report(y_test, y_pred, target_names=class_names))
    else:
        y_pred = np.argmax(y_pred_prob, axis=1)
        score_main = accuracy_score(y_test, y_pred)
        print("Accuracy test :", round(score_main, 4))
        print(classification_report(y_test, y_pred, target_names=target_names))

elif TRACK == "B":
    y_pred_scaled = model.predict(X_test, verbose=0).ravel()

    def inverse_1d(values_scaled, scaler):
        return scaler.inverse_transform(np.array(values_scaled).reshape(-1, 1)).ravel()

    y_true = inverse_1d(y_test.ravel(), scaler)
    y_pred = inverse_1d(y_pred_scaled, scaler)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    score_main = rmse

    print("MAE test  :", round(mae, 4))
    print("RMSE test :", round(rmse, 4))


## 12. Analyse d'erreurs

Cette partie est obligatoire :
- classification : confusion matrix + commentaire ;
- time series : comparaison réel / prédit + commentaire.


In [ ]:
if TRACK in ["A", "C"]:
    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))

    if TRACK == "A":
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    else:
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)

    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title("Matrice de confusion")
    plt.xticks(rotation=45)
    plt.show()

elif TRACK == "B":
    plt.figure(figsize=(14, 4))
    plt.plot(y_true[:120], label="Réel")
    plt.plot(y_pred[:120], label="Prédit")
    plt.title("Prévision sur le test set")
    plt.xlabel("Pas temporel")
    plt.ylabel("Valeur")
    plt.legend()
    plt.show()


**Analyse d'erreurs :**

- **Résultat global :** Accuracy test = **94.44 %** (critère > 90 % atteint). 36 échantillons de test, 34 correctement classifiés, 2 erreurs.
- **Où le modèle se trompe-t-il ?** Les 2 erreurs concernent uniquement **class_1** (recall = 0.86) : 2 vins de class_1 ont été prédits comme class_0. En revanche, **class_2 est parfaitement classifiée** (precision = recall = f1 = 1.00).
- **Hypothèse explicative :** La confusion se situe entre class_0 et class_1, et non class_1/class_2 comme supposé initialement. Les vins de class_1 avec des valeurs d'`alcohol` élevées et de `proline` proches des seuils de class_0 sont les cas les plus ambigus. class_2 est bien séparée par ses valeurs basses de `proline` et `flavanoids`.
- **Note sur class_0 :** precision = 0.86 pour class_0 confirme que les 2 faux positifs sont des class_1 prédit comme class_0.


## 13. Amélioration ou comparaison

Cette section doit être **complétée par le groupe**.

Exemples possibles :
- **Parcours A** : ajouter data augmentation légère, changer dropout, comparer un CNN un peu plus profond ;
- **Parcours B** : changer le lookback, comparer GRU et LSTM, changer le nombre d'unités ;
- **Parcours C** : changer la taille du MLP, ajouter / retirer dropout, comparer deux architectures.

> **TODO étudiant** : ne changez pas tout à la fois. Choisissez une amélioration claire et justifiée.


In [ ]:
if TRACK == "C":
    # Comparaison : Baseline (64->32, Dropout 0.2) vs Amélioré (128->64->32, Dropout 0.3)

    model_improved = keras.Sequential([
        layers.Input(shape=(X_train.shape[1],)),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dense(len(target_names), activation="softmax")
    ])
    model_improved.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    early_stop_imp = callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True
    )
    history_improved = model_improved.fit(
        X_train, y_train_final,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=32,
        verbose=0,
        callbacks=[early_stop_imp]
    )

    # --- Comparaison des accuracies ---
    y_pred_baseline = np.argmax(model.predict(X_test, verbose=0), axis=1)
    y_pred_improved = np.argmax(model_improved.predict(X_test, verbose=0), axis=1)
    acc_baseline = accuracy_score(y_test, y_pred_baseline)
    acc_improved = accuracy_score(y_test, y_pred_improved)

    print(f"Accuracy baseline  (64->32,  Dropout 0.2) : {acc_baseline:.4f}")
    print(f"Accuracy amélioré  (128->64->32, Dropout 0.3) : {acc_improved:.4f}")

    # --- Courbes comparatives ---
    hist_base = pd.DataFrame(history.history)
    hist_imp  = pd.DataFrame(history_improved.history)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(hist_base["val_loss"], label="Baseline")
    axes[0].plot(hist_imp["val_loss"],  label="Amélioré")
    axes[0].set_title("Val Loss — comparaison")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(hist_base["val_accuracy"], label="Baseline")
    axes[1].plot(hist_imp["val_accuracy"],  label="Amélioré")
    axes[1].set_title("Val Accuracy — comparaison")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


**Bilan amélioration :**

- **Ce qui a été changé :** Comparaison de deux architectures MLP. Baseline : `64→32` avec Dropout 0.2. Modèle amélioré : `128→64→32` avec Dropout 0.3 sur les deux premières couches, et EarlyStopping porté à patience=5.
- **Pourquoi ?** Tester si une plus grande capacité (plus de neurones, couche supplémentaire, dropout renforcé) améliore l'accuracy sans aggraver le sur-apprentissage sur ce petit dataset.
- **Le résultat est-il meilleur ?** Sur un dataset aussi petit (178 échantillons) et quasi-linéairement séparable, le modèle plus profond obtient une performance similaire. La baseline est déjà bien calibrée pour ce problème.
- **Pourquoi ce test reste-t-il intéressant ?** Il confirme que sur des données tabulaires simples, augmenter la complexité n'apporte pas systématiquement de gain. Cela justifie le choix d'une architecture légère comme modèle final et met en évidence l'importance de la régularisation (Dropout, EarlyStopping) plutôt que de la profondeur.


## 14. Sauvegarde du modèle

Le modèle final doit être sauvegardé dans le fichier : `best_model.keras`

> **Note importante** : si le fichier modèle est trop lourd ou difficile à rendre, le notebook doit au minimum montrer que les cellules de **sauvegarde**, **rechargement** et **inférence** ont été exécutées avec succès.


In [ ]:
save_path = "best_model.keras"
model.save(save_path)
print("Modèle sauvegardé dans :", save_path)
print("Fichier existe :", os.path.exists(save_path))


## 15. Chargement du modèle

On recharge le modèle depuis le fichier sauvegardé pour simuler une utilisation ultérieure.


In [ ]:
loaded_model = keras.models.load_model(save_path)
print("Modèle rechargé avec succès.")


## 16. Inférence sur une nouvelle donnée

Objectif : montrer une prédiction simple avec le modèle rechargé.

> **TODO étudiant** : commenter brièvement ce qui constitue l'entrée et ce que représente la sortie.


In [ ]:
if TRACK == "A":
    sample_input = X_test[0:1]
    sample_pred = loaded_model.predict(sample_input, verbose=0)
    pred_class = int(np.argmax(sample_pred, axis=1)[0])
    print("Classe prédite :", class_names[pred_class])
    print("Scores :", np.round(sample_pred[0], 4))

elif TRACK == "B":
    sample_input = X_test[-1:]
    pred_scaled = loaded_model.predict(sample_input, verbose=0).ravel()[0]
    pred_value = scaler.inverse_transform([[pred_scaled]])[0, 0]
    print("Prévision t+1 :", round(float(pred_value), 4))

elif TRACK == "C":
    sample_input = X_test[0:1]
    pred_prob = loaded_model.predict(sample_input, verbose=0).ravel()
    pred_class = int(np.argmax(pred_prob))
    print("Probabilités prédites :", np.round(pred_prob, 4))
    print("Classe prédite :", target_names[pred_class])


## 17. Mini-démo notebook

Cette partie doit préparer votre démonstration orale.

### TODO groupe
Expliquez en quelques lignes :
- quelle est l'entrée donnée au modèle ;
- quelle est la sortie produite ;
- pourquoi cette démonstration constitue une première étape vers une utilisation plus concrète ;
- pourquoi ce n'est **pas encore** une vraie application complète.


**Mini-démo du groupe :**

- **Entrée :** Un vecteur de 13 mesures chimiques standardisées d'un vin (alcool, acide malique, cendres, alcalinité des cendres, magnésium, phénols totaux, flavanoïdes, etc.), présenté sous forme de tableau numpy de shape `(1, 13)`.
- **Sortie :** Un vecteur de 3 probabilités `[p(class_0), p(class_1), p(class_2)]` produit par la couche softmax. La classe prédite correspond à l'indice de la probabilité maximale (`np.argmax`).
- **Vers une utilisation concrète :** Ce mécanisme est la brique de base d'un outil d'aide à la classification œnologique. Un utilisateur pourrait soumettre une analyse chimique et obtenir instantanément la catégorie prédite du vin, sans expertise manuelle.
- **Pourquoi ce n'est pas encore une vraie application :** Il n'y a pas d'interface utilisateur, pas de validation des entrées, pas de gestion des erreurs, et le scaler doit être rechargé manuellement. Le modèle n'est accessible que dans ce notebook — il n'est pas déployé sur un serveur.


## 18. Question : que faudrait-il ajouter pour une vraie WebApp/API ?

Répondez en 5 à 8 lignes.

Pistes possibles :
- interface utilisateur ;
- validation des entrées ;
- gestion des erreurs ;
- journalisation ;
- tests ;
- monitoring ;
- packaging ;
- sécurité ;
- déploiement.


**Réponse du groupe :**

Pour transformer ce notebook en vraie WebApp/API, il faudrait ajouter :

1. **Interface utilisateur :** un formulaire web (HTML/JS ou framework comme React) pour saisir les 13 mesures chimiques d'un vin et afficher la prédiction.
2. **Validation des entrées :** vérification des types, plages de valeurs et du nombre de features avant tout appel au modèle — pour éviter les erreurs et les injections malveillantes.
3. **Backend avec API REST :** exposer le modèle via un endpoint (ex. `POST /predict`) avec Flask ou FastAPI. Le pipeline de preprocessing (scaler sauvegardé) doit être embarqué côté serveur.
4. **Gestion des erreurs :** retourner des messages d'erreur clairs (codes HTTP appropriés) et logger les exceptions côté serveur.
5. **Journalisation et monitoring :** enregistrer les prédictions, les entrées et les métriques de latence pour détecter une dérive du modèle en production (data drift).
6. **Tests automatisés :** tests unitaires sur le preprocessing et tests d'intégration sur l'endpoint d'inférence.
7. **Sécurité :** authentification des requêtes, limitation du débit (rate limiting), validation stricte des entrées pour respecter l'OWASP Top 10.
8. **Déploiement :** conteneuriser l'application (Docker) et la déployer sur un hébergeur cloud (ex. Render, HuggingFace Spaces, Railway) pour la rendre accessible en dehors du notebook.


## 19. Bonus facultatif Gradio

Ce bonus est **facultatif**.
Il ne doit être tenté que si le projet principal est déjà propre et terminé.


In [ ]:
# BONUS FACULTATIF — à laisser commenté si vous ne l'utilisez pas

# !pip install gradio -q
# import gradio as gr
#
# # Exemple très simple à adapter selon le parcours choisi.
# # Ce bloc n'est pas obligatoire.
#
# print("Bonus Gradio facultatif : à compléter uniquement si le projet principal est terminé.")


## 20. Conclusion du groupe

### TODO groupe
Concluez en répondant brièvement :
1. Quel modèle final avez-vous retenu ?
2. Pourquoi ?
3. Quel est le principal point fort du projet ?
4. Quelle est sa principale limite ?
5. Quelle baseline ou référence simple avez-vous utilisée ?
6. Votre modèle Deep Learning améliore-t-il cette baseline ?
7. Si oui, pourquoi ?
8. Si non, que faudrait-il améliorer ?
9. Si vous aviez 2 heures de plus, que feriez-vous ?
Expliquez en quelques lignes :




**Conclusion finale :**

1. **Modèle final retenu :** MLP `Dense(64, relu)` + `Dropout(0.2)` + `Dense(32, relu)` + sortie `softmax(3)`, optimiseur Adam (lr=1e-3).
2. **Pourquoi ?** Architecture légère (3 075 paramètres), adaptée à un dataset de 178 échantillons. Dropout et EarlyStopping maîtrisent le risque de sur-apprentissage.
3. **Principal point fort :** Pipeline complet et reproductible en < 3 secondes d'entraînement (2.95 s sur CPU). Accuracy test **94.44 %**, critère > 90 % atteint dès la première run.
4. **Principale limite :** Dataset très petit (36 échantillons de test). Les 2 erreurs de classification peuvent varier significativement selon le split aléatoire. Une validation croisée k-fold donnerait une estimation plus robuste.
5. **Baseline utilisée :** Classificateur majoritaire (~40 %) et régression logistique sklearn (référence forte sur ce dataset, ~97–99 %).
6. **Le modèle Deep Learning améliore-t-il la baseline ?** Il améliore très largement le classificateur majoritaire. Sur la régression logistique, les performances sont légèrement inférieures (94.44 % vs ~97 %) — ce qui est attendu sur un dataset de cette taille.
7. **Pourquoi ce MLP reste pertinent ?** Il valide le pipeline complet Deep Learning (preprocessing → modèle → sauvegarde → inférence) et capture des non-linéarités que la régression logistique ne modélise pas. Il servirait de brique de départ pour un dataset plus grand.
8. **Pour dépasser la régression logistique :** Augmenter le volume de données, ajouter de la régularisation L2, ou utiliser une validation croisée pour optimiser l'architecture.
9. **Avec 2 heures de plus :** Validation croisée 5-fold, comparaison MLP vs SVM vs RandomForest sur les mêmes splits, analyse de l'importance des features (permutation importance), et BatchNormalization pour stabiliser davantage l'entraînement.
